# 6. От углового конуса к произвольно ориентированному сегменту

Это учебный прототип в `lighthit.experimental.cone_segment`, ещё не быстрый
G4-процессор. Детектор остаётся точечным и изотропным. Геометрия, скорость
частицы, время и черенковский угол каждого сегмента сохраняются явно.

In [ ]:
from pathlib import Path
import sys, time, platform
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Notebook can be started from the repo root or notebooks/course.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/lighthit').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Start this notebook inside the LightHit repository')
sys.path.insert(0, str(ROOT / 'src'))
from lighthit import Medium, SolverSettings, PointGreenSolver

# Explicit public test medium. No private provider is imported.
medium = Medium(0.04, 0.05, 0.7, 1.35, 450.0, 'course-synthetic')
np.set_printoptions(precision=7, suppress=True)
print('Python:', sys.executable)
print('Platform:', platform.platform())

## 6.1. Угловой интеграл конуса

Для направления частицы $\mathbf u$, $\beta=v_p/c_0$, фазового показателя
$n_{\rm ph}$ и $\beta n_{\rm ph}>1$:
$$p_C(\mathbf s)=\frac{\delta(\mathbf u\cdot\mathbf s-\mu_C)}{2\pi},
\qquad\mu_C=\frac1{\beta n_{\rm ph}}.$$
В отличие от $v_p$, скорость фотонов равна $v_g=c_0/n_g$.
По теореме сложения
$$\int p_C(\mathbf s)P_\ell(\widehat{\mathbf r}\cdot\mathbf s)d\Omega
=P_\ell(\mu_C)P_\ell(\widehat{\mathbf r}\cdot\mathbf u).$$
Азимут конуса больше не нужно семплировать при применении мультиполей.

In [ ]:
from scipy.special import eval_legendre
muC=.75;nu=.3;phi=2*np.pi*np.arange(1024)/1024
cos_to_r=muC*nu+np.sqrt(1-muC**2)*np.sqrt(1-nu**2)*np.cos(phi)
for ell in range(12):
    numerical=np.mean(eval_legendre(ell,cos_to_r))
    analytic=eval_legendre(ell,muC)*eval_legendre(ell,nu)
    assert abs(numerical-analytic)<2e-14
print('Cone identity passed through degree 11')

## 6.2. Пространство и время вдоль сегмента

Пусть начало $\mathbf x_0$, длина $h$, направление $\mathbf u$, постоянный
выход $y$ фотонов/м и начало времени $t_0$. Тогда
$\mathbf x(a)=\mathbf x_0+a\mathbf u$, $t(a)=t_0+a/v_p$, $0\le a<h$.
Для ОМ $d$:
$$\mathbf r_d(a)=\mathbf R_d-\mathbf x(a),\quad r_d=|\mathbf r_d|,\quad
\nu_d(a)=\widehat{\mathbf r}_d(a)\cdot\mathbf u.$$
Рассеянный спектр:
$$N_d^{(p)}(\omega)=y\int_0^h da\,e^{i\omega(t_0+a/v_p)}
\sum_{\ell=0}^J\mathcal R_\ell^{(p)}(r_d(a),\omega)
P_\ell(\mu_C)P_\ell(\nu_d(a)).$$
Здесь $p=1,\ge2$; первый порядок в этом прототипе тоже использует конечный
$L$. Точечный `PointGreenSolver` имеет иную композицию первого порядка;
это различие явно сохраняется в тестах и подписи результата.

## 6.3. Прямой свет — аналитический корень

Положим $z=(\mathbf R_d-\mathbf x_0)\cdot\mathbf u$,
$b=|\mathbf R_d-\mathbf x_0-z\mathbf u|>0$,
$s_C=\sqrt{1-\mu_C^2}$. Луч конуса попадает в точку при
$$a_*=z-b\mu_C/s_C,\qquad R_*=b/s_C.$$
Если $0\le a_*<h$,
$$Q_d^{(0)}=\frac{y e^{-\mu_t R_*}}{2\pi b s_C},\qquad
N_d^{(0)}(\omega)=Q_d^{(0)}e^{i\omega(t_0+a_*/v_p+R_*/v_g)}.$$
В противном случае прямой вклад равен нулю. Полуоткрытый отрезок предотвращает
двойной учёт точного корня на границе двух соседних сегментов.

In [ ]:
from dataclasses import replace
from lighthit.cache import CacheGrid,ResponseCache
from lighthit.experimental.cone_segment import ConeSegment,segment_spectrum
segment=ConeSegment((0.,0.,0.),(0.,0.,1.),8.,.99,1.34,1.)
positions=np.array([[10.,0.,14.],[-11.,4.,12.]])
cache=ResponseCache.build(medium,SolverSettings(32,80,4.,.05,8),
    CacheGrid.geometric(9.,25.,20,[0.,.02,.1]),radial_phase='flight')
t0=time.perf_counter();response=segment_spectrum(cache,segment,positions,longitudinal_order=24)
print('Query [s]:',time.perf_counter()-t0)
print('Charges by order (0,finite-L 1,>=2):\n',response[0].real)

In [ ]:
from scipy.spatial.transform import Rotation
Q=Rotation.from_rotvec([.2,-.4,.6]).as_matrix();shift=np.array([11.,-8.,3.])
turned=replace(segment,start_m=tuple(shift),direction=tuple(Q@np.array(segment.direction)))
same=segment_spectrum(cache,turned,positions@Q.T+shift,longitudinal_order=24)
np.testing.assert_allclose(same,response,rtol=2e-12,atol=1e-18)
a=replace(segment,length_m=4.)
b=replace(segment,start_m=(0.,0.,4.),length_m=4.,start_time_ns=4/segment.speed_m_per_ns)
split=segment_spectrum(cache,a,positions,longitudinal_order=24)+segment_spectrum(cache,b,positions,longitudinal_order=24)
print('Split residual / max signal:',np.max(np.abs(split-response))/np.max(np.abs(response)))

## 6.4. Почему короткий шаг не всегда равен точке

Даже при почти постоянной амплитуде вдоль шага продольная фаза даёт
$$h\operatorname{sinc}\left[\frac{\omega h}{2}
\left(\frac1{v_p}-\frac{\widehat{\mathbf r}\cdot\mathbf u}{v_g}\right)\right].$$
Поэтому $h/r\ll1$ недостаточно. Для midpoint нужны также малая фазовая
разность и медленное изменение угловой зависимости/амплитуды.

## Задания и следующий шаг

Уточнить продольную квадратуру и радиальный кэш независимо. Сложить два
сегмента с разными направлениями и временами. Показать, какие корреляции
теряются при замене этой суммы одним средним направлением.

Для G4 нужны истинная длина, хорда, $t_0,t_1$, $\beta$ и спектральный вес
каждого шага. Нельзя молча заменять эти величины одной скоростью и одной осью.
Контракт источника и план общего ОМ находятся в `docs/research/tracks-and-showers.md`.